# M5 (Walmart) Datensatz

- **Quelle**: Lehrstuhl für Logistik und Supply Chain (ursprünglich M5 Accuracy Competition auf Kaggle: https://www.kaggle.com/competitions/m5-forecasting-accuracy)
- **Zielvariable**: Nachfrage (`demand`)
- **Frequenz**: täglich
- **Zeitraum**: 26.02.2011 bis 19.06.2016 (1941 Tage)
- **Granularität**: unterste Ebene (item_id, store_id, täglich)
---
- Anzahl einzigartiger **Artikel**: 10
- Anzahl einzigartiger **Filialen**: 10
- Anzahl einzigartiger Kombinationen aus **Artikel** und **Filialen** (= ID): 100

## 1. Data Collection

Dieser Datensatz wurde vom Lehrstuhl für Logistik und Supply Chain bereitgestellt. Ursprünglich stammt er aus einem Kaggle-Wettbewerb.

## 2. Data Overview & Description

Eine detaillierte Beschreibung finden Sie in der Dokumentation: https://katja19.github.io/multi-period-forecasting/0_data_preprocessing/m5_data/#2-data-description

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
# For plotting and saving figures
import pandas as pd
import plotly.graph_objects as go
import os
import json
import plotly.io as pio
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Set display options for pandas for full visibility of DataFrame contents
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# get the path
data_path = Path.cwd().parents[1] / 'data' / 'raw' / 'dataM5.csv'

# Load the dataset
m5_df = pd.read_csv(data_path)

In [ ]:
m5_df.head(5)

In [ ]:
m5_df.shape

In [ ]:
m5_df.dtypes

### checking date column

In [ ]:
# Ensure 'date' is datetime
m5_df['date'] = pd.to_datetime(m5_df['date'])

# Gesamtzeitraum
overall_min, overall_max = m5_df['date'].agg(['min', 'max'])
print(f"Overall: {overall_min.date()} to {overall_max.date()} ({(overall_max - overall_min).days + 1} days)") # +1 to include both start and end dates

# Funktion für min, max & Dauer pro Label
for label in ['train', 'test']:
    subset = m5_df[m5_df['label'] == label]['date']
    if not subset.empty:
        min_d, max_d = subset.min(), subset.max()
        duration = (max_d - min_d).days + 1  # Include both start and end dates!!!
        print(f"{label.capitalize()} horizon: {min_d.date()} to {max_d.date()} ({duration} days)")

In [ ]:
# check if there are any missing dates between the min and max dates
def check_missing_dates(df, date_col):
    all_dates = pd.date_range(start=df[date_col].min(), end=df[date_col].max())
    missing_dates = all_dates.difference(df[date_col])
    return missing_dates
missing_dates = check_missing_dates(m5_df, 'date')
if not missing_dates.empty:
    print(f"Missing dates found: {missing_dates.tolist()}")
else:
    print("No missing dates found in the dataset.")

### describe, info, unique values

In [ ]:
m5_df.describe(include='all') # all = numeric and object columns, default is numeric only

In [ ]:
m5_df.info()

In [ ]:
# ckeck for unique values
unique_values = m5_df.nunique()
print("Unique values in each column:")
unique_values

### amount of unique items and stores

In [ ]:
# count how many columns start with 'item_'
item_columns = [col for col in m5_df.columns if col.startswith('item_')]
print(f"Number of item columns: {len(item_columns)}")
# Count how many columns start with 'store_'
store_columns = [col for col in m5_df.columns if col.startswith('store_')]
print(f"Number of store columns: {len(store_columns)}")

### Scaling Value

#### Wie wird die Nachfrage (demand) skaliert?

Antwort:

$demand (scaled) = \frac{demand\_original (reconstructed)}{scalingValue}$

(gruppiert je id (item_store Kombination))

In [ ]:
scaling_value = m5_df['scalingValue'].unique()
scaling_value

In [ ]:
m5_df['demand_original_reconstructed'] = m5_df['scalingValue'] * m5_df['demand']
print(m5_df["demand_original_reconstructed"].describe() )

In [ ]:
print(m5_df["demand"].describe() )

In [ ]:
m5_df[["demand", "scalingValue", "demand_original_reconstructed"]].head(10)

In [ ]:
m5_df.groupby("id")["scalingValue"].nunique().value_counts()

#### Wie wurde scalingValue berechnet?

Antwort: Der scalingValue is identisch bis auf in fünf Fällen wie der max(demand_original_reconstructed) by ID.

In [ ]:
agg_df = m5_df.groupby("id").agg({
    "demand_original_reconstructed": ["max", "mean", "std", "median"],
    "scalingValue": "first"  # Da pro ID konstant
}).reset_index()

# Spalten umbenennen
agg_df.columns = ["id", "max", "mean", "std", "median", "scalingValue"]

In [ ]:
print(agg_df.corr(numeric_only=True)["scalingValue"])

In [ ]:
agg_df["diff_max"] = np.abs(agg_df["scalingValue"] - agg_df["max"])
agg_df["diff_mean"] = np.abs(agg_df["scalingValue"] - agg_df["mean"])
agg_df["diff_median"] = np.abs(agg_df["scalingValue"] - agg_df["median"])

agg_df[["id", "scalingValue", "max", "mean", "median", "diff_max", "diff_mean", "diff_median"]].sort_values("diff_max").head(10)

In [ ]:
# check if diff_max is something other than 0
agg_df[agg_df["diff_max"] != 0].shape[0] == 0  # besst case: all diff_max are 0, meaning scalingValue is equal to max(demand_original_reconstructed) for all IDs, but this is not the case here

In [ ]:
# get all unique values of diff_max
unique_diff_max = agg_df["diff_max"].unique()
unique_diff_max
# count how often each unique value occurs
diff_max_counts = agg_df["diff_max"].value_counts()
print("Counts of unique diff_max values:")
print(diff_max_counts)

In [ ]:
agg_df[agg_df["diff_max"] != 0][["id", "scalingValue", "max", "diff_max"]]

## 3. Data Cleaning

### Missing Values

In [ ]:
#check for missing values in the dataset
missing = m5_df.isna().sum() # isna() is more robust than isnull()
print(missing[missing > 0] if (missing > 0).any() else "No missing values found.")

### Duplicates

In [ ]:
#chek for duplicate rows in the dataset
print(f"Duplicate rows found: {m5_df.duplicated().sum()}" if m5_df.duplicated().any() else "No duplicate rows found.")

### Outliners

In [ ]:
m5_df.head()

In [ ]:
# cols that are sutable for boxplots
boxplot_vars = [col for col in m5_df.columns if m5_df[col].nunique() > 2 and m5_df[col].dtype in ['bool', 'int', 'float']]
# exclude columns that are not suitable for boxplots 'weekyear', 'month', 'year', 'yearIndex'
boxplot_vars = [col for col in boxplot_vars if col not in ['weekyear', 'month', 'year', 'dayIndex']]
boxplot_vars

In [ ]:
# splitting boxplot vars into to case else the json whould be to big
print(len(boxplot_vars))
boxplot_vars_part_1 = []
boxplot_vars_part_2 = []
for var in boxplot_vars:
    if "14" in var or "28" in var:
        boxplot_vars_part_2.append(var)
    else:
        boxplot_vars_part_1.append(var)
print(len(boxplot_vars_part_1))
print(len(boxplot_vars_part_2))
        

In [ ]:
def save_boxplots(fig_box, name):
    # JSON speichern
    fig_json = pio.to_json(fig_box, pretty=True)
    # Speicherpfad setzen
    output_dir = r"..\..\docs\assets\json\m5"
    output_file = f"{name}.json"
    output_path = os.path.join(output_dir, output_file)

    # Sicherstellen, dass der Ordner existiert
    os.makedirs(output_dir, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(fig_json)
    print(f"Plotly JSON gespeichert unter {output_path}")

In [ ]:
# Create boxplot for each variable in boxplot_vars
# Using Plotly for interactive boxplots
def creat_boxplot_and_saving(boxplot_vars, name):
    df = m5_df[boxplot_vars].copy()
    initial_col = df.columns[0]

    fig_box = go.Figure()

    for col in df.columns:
        fig_box.add_trace(go.Box(
            y=df[col].tolist(),  # WICHTIG: tolist(), damit keine bdata im JSON entsteht
            name=col,
            visible=(col == initial_col)
        ))

    dropdown_buttons = [
        dict(
            label=col,
            method="update",
            args=[
                {"visible": [c == col for c in df.columns]},
                {
                    "title": f"Boxplot für {col}",
                    "yaxis": {
                        "title": {"text": col},
                        "range": [
                            df[col].min() - (df[col].std() * 0.5),
                            df[col].max() + (df[col].std() * 0.5)
                        ]
                    }
                }
            ]
        )
        for col in df.columns
    ]

    fig_box.update_layout(
        updatemenus=[
            dict(
                active=0,
                buttons=dropdown_buttons,
                x=1.15,
                xanchor='left',
                y=1.1,
                yanchor='top'
            )
        ],
        title=f"Boxplot für {initial_col}",
        yaxis_title=initial_col,
        showlegend=False,
        height=500
    )

    # Plot anzeigen
    fig_box.show()
    
    save_boxplots(fig_box, name)

In [ ]:
creat_boxplot_and_saving(boxplot_vars_part_1, "m5_boxplot_multiple_part_1")
creat_boxplot_and_saving(boxplot_vars_part_2, "m5_boxplot_multiple_part_2")

### Inconsistency

- Nur ein Teil der Boolischen Variablen (Columns) wurden binär encoded. Der Einheitlichkeit werden die anderen auch binär encoded.
- Anpassung der datentypen für semantische korrektheit, speicher und performance vorteile, modellverhalten.

In [ ]:
# scalingValue
# Check if all values in the float column are whole numbers
can_convert = (m5_df['scalingValue'] % 1 == 0).all()
if can_convert:
    print("All values in 'scalingValue' are whole numbers. Safe to convert to int.")
else:
    print("Not all values in 'scalingValue' are whole numbers. Cannot safely convert to int.")
    #show these values

# show all values in 'scalingValue' that are not whole numbers
non_whole_values = m5_df[m5_df['scalingValue'] % 1 != 0]['scalingValue'].unique()
if non_whole_values.size > 0:
    print("Values in 'scalingValue' that are not whole numbers:")
    print(non_whole_values)

In [ ]:
# dayIndex: float -> int 
# Check if all values in the float column are whole numbers
can_convert = (m5_df['dayIndex'] % 1 == 0).all()
if can_convert:
    print("All values in 'dayIndex' are whole numbers. Safe to convert to int.")
else:
    print("Not all values in 'dayIndex' are whole numbers. Cannot safely convert to int.")
    # show all values in 'dayIndex' that are not whole numbers
    non_whole_values = m5_df[m5_df['dayIndex'] % 1 != 0]['dayIndex'].unique()
    if non_whole_values.size > 0:
        print("Values in 'dayIndex' that are not whole numbers:")
        print(non_whole_values)
        
# convert dayIndex to int
m5_df['dayIndex'] = m5_df['dayIndex'].astype(int)

In [ ]:
# datetime
# convert date to datetime
m5_df['date'] = pd.to_datetime(m5_df['date'])

In [ ]:
# bool cols: is_...
# check if the columns starting with 'is_' are all whole numbers
is_cols = m5_df.filter(like='is_')
print(f"Columns starting with 'is_': {is_cols.columns.tolist()}")
for col in is_cols.columns:
    can_convert = (m5_df[col] % 1 == 0).all()
    if can_convert:
        print(f"All values in '{col}' are whole numbers. Safe to convert to int.")
    else:
        print(f"Not all values in '{col}' are whole numbers. Cannot safely convert to int.")
        
# change the type of the columns starting with 'is_' to int
for col in is_cols.columns:
    m5_df[col] = m5_df[col].astype(int)

In [ ]:
# one-hot encodete Variablen von bool zu int (später label encoding für weekday, month, year, damit beide Datensätze gleich sind)
# binar encode all columns starting with 'state_', 'weekday_', 'month_', 'year_', 'item_' or 'store_' from bool to int
binary_cols = m5_df.columns[m5_df.columns.str.startswith(('state_', 'weekday_', 'month_', 'year_', 'item_', 'store_'))]
for col in binary_cols:
    if m5_df[col].dtype == 'bool':
        m5_df[col] = m5_df[col].astype(int)
    else:
        print(f"Column '{col}' is not of type bool, skipping conversion.")

## 4. EDA: Visualisation

In [ ]:
# Sicherstellen, dass Zeitgruppen existieren: adding year, yearMonth, and yearWeek columns for grouping
m5_df['year'] = m5_df['date'].dt.year
m5_df['yearMonth'] = m5_df['date'].dt.strftime('%Y-%m')
m5_df['yearWeek'] = m5_df['date'].dt.strftime('%G-%V')

In [ ]:
# Plot speichern
def save_plotly_figure_2(fig, filename, groupedBy):
    filepath = os.path.join("..", "..", "docs", "assets", "json", "m5", groupedBy, f"m5_{filename}.json")
    fig_json = pio.to_json(fig)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(json.loads(fig_json), f, indent=4)

### Total Demand

In [ ]:
# Total
# Helferfunktion zum Erstellen und Speichern eines Linienplots
def create_and_save_line_plot(x_values, y_values, title, xaxis_title, yaxis_title, 
                              filename, groupedBy, outliers=None, xaxis_range=None):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=x_values, y=y_values, mode='lines', name=title))

    if outliers is not None:
        fig.add_trace(go.Scatter(
            x=outliers.index.strftime('%Y-%m-%d').tolist(),
            y=outliers.values.tolist(),
            mode='markers',
            name='Outliers (IQR method)',
            marker=dict(color='red', size=5),
            
        ))

    fig.update_layout(
        title=title,
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        autosize=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1)
    )
    
    fig.update_xaxes(type='date', range=xaxis_range,tick0='2011-01-01', dtick='M6', tickformat='%b %Y')
    
    # hovertemplate:
    fig.update_traces(
        hovertemplate='%{x|%Y-%m-%d}<br>Total demand: %{y}'
    )

    fig.show()
    save_plotly_figure_2(fig, filename, groupedBy)

In [ ]:
# ========== TÄGLICHE AGGREGATION ========== #
daily = m5_df.groupby('date')['demand_original_reconstructed'].sum()
xaxis_range=['2011-01-01', '2016-07-01']  # <-- geändert

# IQR-basiertes Outlier-Detection
Q1, Q3 = daily.quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = daily[(daily < Q1 - 1.5 * IQR) | (daily > Q3 + 1.5 * IQR)]

print(type(daily.index))
print(daily.index[:5])

create_and_save_line_plot(
    x_values=daily.index.strftime('%Y-%m-%d').tolist(),
    y_values=daily.values.tolist(),
    title='Total demand per day with outliers',
    xaxis_title='Date',
    yaxis_title='Total demand',
    filename='total_demand_day',
    groupedBy='totalDemand',
    outliers=outliers,
    xaxis_range=xaxis_range
)

# ========== WÖCHENTLICHE AGGREGATION ========== #
weekly = m5_df.groupby('yearWeek')['demand_original_reconstructed'].sum()
weekly.index = pd.to_datetime(weekly.index + '-1', format='%G-%V-%u')

create_and_save_line_plot(
    x_values=weekly.index.strftime('%Y-%m-%d').tolist(),
    y_values=weekly.values.tolist(),
    title='Total demand per week',
    xaxis_title='Week',
    yaxis_title='Total demand',
    filename='total_demand_week',
    groupedBy='totalDemand',
    xaxis_range=xaxis_range
)

# ========== MONATLICHE AGGREGATION ========== #
monthly = m5_df.groupby('yearMonth')['demand_original_reconstructed'].sum()
monthly.index = pd.to_datetime(monthly.index + '-01', format='%Y-%m-%d')

create_and_save_line_plot(
    x_values=monthly.index.strftime('%Y-%m-%d').tolist(),
    y_values=monthly.values.tolist(),
    title='Total demand per month',
    xaxis_title='Month',
    yaxis_title='Total demand',
    filename='total_demand_month',
    groupedBy='totalDemand',
    xaxis_range=xaxis_range
)

# ========== JÄHRLICHE AGGREGATION ========== #
yearly = m5_df.groupby(m5_df['date'].dt.year)['demand_original_reconstructed'].sum()
yearly.index = pd.to_datetime(yearly.index, format='%Y')

create_and_save_line_plot(
    x_values=yearly.index.strftime('%Y').tolist(),
    y_values=yearly.values.tolist(),
    title='Total demand per year',
    xaxis_title='Year',
    yaxis_title='Total demand',
    filename='total_demand_year',
    groupedBy='totalDemand',
    xaxis_range=xaxis_range
)

### Demand per Item

In [ ]:
# print every column name starting with 'item_'
item_columns = [col for col in m5_df.columns if col.startswith('item_')]
store_columns = [col for col in m5_df.columns if col.startswith('store_')]

# make columns item_id and store_id out of the one-hot encoded columns
m5_df['item_id'] = m5_df[item_columns].idxmax(axis=1)
m5_df['store_id'] = m5_df[store_columns].idxmax(axis=1) #m5_df['item'] = m5_df['item'].str.replace('item_', '', regex=False)

In [ ]:
m5_df.head()

In [ ]:
# Aggregating the total demand per day and item
total_demand_per_item_per_day = m5_df.groupby(['date'] + item_columns)['demand_original_reconstructed'].sum().reset_index()

# Remove the time from the date (only keep the date part)
total_demand_per_item_per_day['date'] = total_demand_per_item_per_day['date'].dt.date

# Ensure date column is datetime for formatting later
total_demand_per_item_per_day['date'] = pd.to_datetime(total_demand_per_item_per_day['date'], errors='coerce')
total_demand_per_item_per_day.dropna(subset=['date'], inplace=True)

# Create Plotly figure
fig = go.Figure()

# Add one trace per item
for item in item_columns:
    filtered = total_demand_per_item_per_day[total_demand_per_item_per_day[item] > 0]
    fig.add_trace(go.Scatter(
        x=filtered['date'].dt.strftime('%Y-%m-%d').tolist(),
        y=filtered['demand_original_reconstructed'].tolist(),
        mode='lines',
        name=item
    ))

# Update layout
fig.update_layout(
    title='Total demand per item per day',
    xaxis_title='Date',
    yaxis_title='Demand',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.4,
        xanchor='center',
        x=0.5
    )
)

# Show and save
fig.show()
#save_plotly_figure_2(fig, 'demand_per_item_day', 'perItem')

In [ ]:
# === 3. Nachfrage je Woche + Item aggregieren
temp2 = m5_df.groupby(['yearWeek'] + item_columns)['demand_original_reconstructed'].sum().reset_index()

# === 4. Durchschnittsnachfrage je Item (nur wo Item==1) zur Sortering der Items für Farbskala
avg_demand_per_item = {
    item: temp2.loc[temp2[item] == 1, 'demand_original_reconstructed'].mean()
    for item in item_columns
}

# === 5. Items nach durchschnittlicher Nachfrage sortieren (absteigend)
sorted_items = sorted(avg_demand_per_item, key=avg_demand_per_item.get, reverse=True)

# === 6. Nachfrage je Woche + Item (sortierte Spaltenreihenfolge)
total_demand_per_item_per_yearWeek = m5_df.groupby(['yearWeek'] + sorted_items)['demand_original_reconstructed'].sum().reset_index()

# === 7. Gesamtnachfrage je Item
item_total_demand = {
    item: total_demand_per_item_per_yearWeek.loc[total_demand_per_item_per_yearWeek[item] > 0, 'demand_original_reconstructed'].sum()
    for item in sorted_items
}

# === 8. Farbskala (Turbo, von rot → blau je nach Nachfrage)
color_scale_name = "Turbo"
color_positions = np.linspace(1, 0, len(sorted_items))  # Reversed!
color_scale = px.colors.sample_colorscale(color_scale_name, color_positions)

# === 9. Plot erstellen
fig = go.Figure()

for i, item in enumerate(sorted_items):
    df_item = total_demand_per_item_per_yearWeek.copy()
    df_item = df_item[df_item[item] > 0]

    # yearWeek → datetime (Montag der Kalenderwoche)
    df_item['yearWeek'] = pd.to_datetime(df_item['yearWeek'] + '-1', format='%G-%V-%u')

    # Alle anderen Item-Spalten droppen
    drop_cols = [col for col in sorted_items if col != item]
    df_item.drop(columns=drop_cols, inplace=True)

    df_item = df_item.dropna(subset=['demand_original_reconstructed'])

    fig.add_trace(go.Scatter(
        x=df_item['yearWeek'].tolist(),
        y=df_item['demand_original_reconstructed'].astype(float).tolist(),
        mode='lines',
        name=item,
        #name=item.replace("item_", ""),  # Kürzerer Name in der Legende
        line=dict(color=color_scale[i])
    ))

# === 10. Layout
fig.update_layout(
    title='Total demand per item per week',
    xaxis_title='Week',
    yaxis_title='Demand',
    height=600,
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.15, 
        xanchor='center',
        x=0.5
    )
)

# === 11. Anzeigen
fig.show()

#save_plotly_figure_2(fig, 'demand_per_item_per_yearWeek', 'perItem')

In [ ]:
# Aggregiere Nachfrage pro Tag und Item
total_demand_per_item_per_day = m5_df.groupby(['date', 'item_id'])['demand_original_reconstructed'].sum().reset_index()

# Datum bereinigen
total_demand_per_item_per_day['date'] = pd.to_datetime(total_demand_per_item_per_day['date'], errors='coerce')
total_demand_per_item_per_day.dropna(subset=['date'], inplace=True)

# Gesamtnachfrage pro Item berechnen
item_total_demand = total_demand_per_item_per_day.groupby('item_id')['demand_original_reconstructed'].sum()

# Items nach Nachfrage sortieren
sorted_items = item_total_demand.sort_values(ascending=False).index.tolist()

# Min-Max für Farbschema
min_demand = item_total_demand.min()
max_demand = item_total_demand.max()

# Farben generieren (rot = viel, blau = wenig)
color_scale_name = "Turbo"
color_positions = np.linspace(1, 0, len(sorted_items))  # umgedreht: rot = viel
color_scale = px.colors.sample_colorscale(color_scale_name, color_positions)

# Mapping: item → color
item_to_color = {
    item: color_scale[i]
    for i, item in enumerate(sorted_items)
}

# Plotly-Figur
fig = go.Figure()

# Plotten
for item in sorted_items:
    filtered = total_demand_per_item_per_day[total_demand_per_item_per_day['item_id'] == item]
    fig.add_trace(go.Scatter(
        x=filtered['date'],
        y=filtered['demand_original_reconstructed'],
        mode='lines',
        name=item,
        #name=item.replace("store_", ""),  # Kürzerer Name in der Legende
        line=dict(color=item_to_color[item])
    ))

# Layout
fig.update_layout(
    title='Total demand per item per day',
    xaxis_title='Date',
    yaxis_title='Demand',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.4,
        xanchor='center',
        x=0.5
    )
)

# Anzeigen
fig.show()

#save_plotly_figure_2(fig, 'demand_per_item_day', 'perItem')

In [ ]:
# Aggregiere Nachfrage pro Tag und Item
total_demand_per_item_per_day = m5_df.groupby(['date', 'item_id'])['demand_original_reconstructed'].sum().reset_index()

# Datum bereinigen
total_demand_per_item_per_day['date'] = pd.to_datetime(total_demand_per_item_per_day['date'], errors='coerce')
total_demand_per_item_per_day.dropna(subset=['date'], inplace=True)

# Gesamtnachfrage pro Item berechnen
item_total_demand = total_demand_per_item_per_day.groupby('item_id')['demand_original_reconstructed'].sum()

# Items nach Nachfrage sortieren
sorted_items = item_total_demand.sort_values(ascending=False).index.tolist()

# Min-Max für Farbschema
min_demand = item_total_demand.min()
max_demand = item_total_demand.max()

# Farben generieren (rot = viel, blau = wenig)
color_scale_name = "Turbo"
color_positions = np.linspace(1, 0, len(sorted_items))  # umgedreht: rot = viel
color_scale = px.colors.sample_colorscale(color_scale_name, color_positions)

# Mapping: item → color
item_to_color = {
    item: color_scale[i]
    for i, item in enumerate(sorted_items)
}

# Plotly-Figur
fig = go.Figure()

# Plotten
for item in sorted_items:
    filtered = total_demand_per_item_per_day[total_demand_per_item_per_day['item_id'] == item]
    fig.add_trace(go.Scatter(
        x=filtered['date'],
        y=filtered['demand_original_reconstructed'],
        mode='lines',
        name=item,
        #name=item.replace("store_", ""),  # Kürzerer Name in der Legende
        line=dict(color=item_to_color[item])
    ))

# Layout
fig.update_layout(
    title='Total demand per item per day',
    xaxis_title='Date',
    yaxis_title='Demand',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.4,
        xanchor='center',
        x=0.5
    )
)

# Anzeigen
fig.show()

save_plotly_figure_2(fig, 'demand_per_item_day', 'perItem')

### Demand per Store

In [ ]:
# === 3. Nachfrage je Woche + Store aggregieren
temp2 = m5_df.groupby(['yearWeek'] + store_columns)['demand_original_reconstructed'].sum().reset_index()

# === 4. Durchschnittsnachfrage je Store (nur wo Store==1) zur Sortering der Stores für Farbskala
avg_demand_per_store = {
    store: temp2.loc[temp2[store] == 1, 'demand_original_reconstructed'].mean()
    for store in store_columns
}

# === 5. Stores nach durchschnittlicher Nachfrage sortieren (absteigend)
sorted_stores = sorted(avg_demand_per_store, key=avg_demand_per_store.get, reverse=True)

# === 6. Nachfrage je Woche + Store (sortierte Spaltenreihenfolge)
total_demand_per_store_per_yearWeek = m5_df.groupby(['yearWeek'] + sorted_stores)['demand_original_reconstructed'].sum().reset_index()

# === 7. Gesamtnachfrage je Store
store_total_demand = {
    store: total_demand_per_store_per_yearWeek.loc[total_demand_per_store_per_yearWeek[store] > 0, 'demand_original_reconstructed'].sum()
    for store in sorted_stores
}

# === 8. Farbskala (Turbo, von rot → blau je nach Nachfrage)
color_scale_name = "Turbo"
color_positions = np.linspace(1, 0, len(sorted_stores))  # Reversed!
color_scale = px.colors.sample_colorscale(color_scale_name, color_positions)

# === 9. Plot erstellen
fig = go.Figure()

for i, store in enumerate(sorted_stores):
    df_store = total_demand_per_store_per_yearWeek.copy()
    df_store = df_store[df_store[store] > 0]

    # yearWeek → datetime (Montag der Kalenderwoche)
    df_store['yearWeek'] = pd.to_datetime(df_store['yearWeek'] + '-1', format='%G-%V-%u')

    # Alle anderen Store-Spalten droppen
    drop_cols = [col for col in sorted_stores if col != store]
    df_store.drop(columns=drop_cols, inplace=True)

    df_store = df_store.dropna(subset=['demand_original_reconstructed'])

    fig.add_trace(go.Scatter(
        x=df_store['yearWeek'].tolist(),
        y=df_store['demand_original_reconstructed'].astype(float).tolist(),
        mode='lines',
        name=store,
        #name=store.replace("store_", ""),  # Kürzerer Name in der Legende
        line=dict(color=color_scale[i])
    ))

# === 10. Layout
fig.update_layout(
    title='Total demand per store per week',
    xaxis_title='Week',
    yaxis_title='Demand',
    height=600,
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.15,
        xanchor='center',
        x=0.5
    )
)

# === 11. Anzeigen
fig.show()

#save_plotly_figure_2(fig, 'demand_per_store_per_yearWeek', 'perStore')

### deamd per state

In [ ]:
# state list
state_columns = ['state_CA', 'state_TX', 'state_WI'] # Califonia, Taxas, Wisconsin


In [ ]:

# Aggregating the total demand per day and state
total_demand_per_state_per_day = m5_df.groupby(['date'] + state_columns)['demand_original_reconstructed'].sum().reset_index()

# Remove the time from the date (only keep the date part)
total_demand_per_state_per_day['date'] = total_demand_per_state_per_day['date'].dt.date

# Ensure date column is datetime for formatting later
total_demand_per_state_per_day['date'] = pd.to_datetime(total_demand_per_state_per_day['date'], errors='coerce')
total_demand_per_state_per_day.dropna(subset=['date'], inplace=True)

# Create Plotly figure
fig = go.Figure()

# Add one trace per state
for state in state_columns:
    filtered = total_demand_per_state_per_day[total_demand_per_state_per_day[state] > 0]
    fig.add_trace(go.Scatter(
        x=filtered['date'].dt.strftime('%Y-%m-%d').tolist(),
        y=filtered['demand_original_reconstructed'].tolist(),
        mode='lines',
        name=state
    ))

# Setze Hovertemplate abhängig vom Zeit-Modus
hover_fmt = '%{x|%Y-%m-%d}' # for weekYear: '%{x|%Y-W%V}'

fig.update_traces(
    hovertemplate=hover_fmt + '<br>Demand: %{y}'
)

# Update layout
fig.update_layout(
    title=f'Total demand per state per day',
    xaxis_title='Date',
    yaxis_title='Demand',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5
    )
)

# Show and save
fig.show()
save_plotly_figure_2(fig, 'demand_per_state_day', 'perState')

In [ ]:
# Aggregating the total demand per day and state
total_demand_per_state_per_week = m5_df.groupby(['yearWeek'] + state_columns)['demand_original_reconstructed'].sum().reset_index()

# Ensure date column is datetime for formatting later
total_demand_per_state_per_week['yearWeek'] = pd.to_datetime(total_demand_per_state_per_week['yearWeek'], errors='coerce')
total_demand_per_state_per_week.dropna(subset=['yearWeek'], inplace=True)

# Create Plotly figure
fig = go.Figure()

# Add one trace per state
for state in state_columns:
    filtered = total_demand_per_state_per_week[total_demand_per_state_per_week[state] > 0]
    fig.add_trace(go.Scatter(
        x=filtered['yearWeek'].dt.strftime('%Y-%m-%d').tolist(),
        y=filtered['demand_original_reconstructed'].tolist(),
        mode='lines',
        name=state
    ))

# Setze Hovertemplate abhängig vom Zeit-Modus
hover_fmt = '%{x|%Y-W%V}' # 

fig.update_traces(
    hovertemplate=hover_fmt + '<br>Demand: %{y}'
)

# Update layout
fig.update_layout(
    title=f'Total demand per state per week',
    xaxis_title='Date',
    yaxis_title='Demand',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5
    )
)

# Show and save
fig.show()
save_plotly_figure_2(fig, 'demand_per_state_yearWeek', 'perState')

### Demand per Item-Store

In [ ]:
# === 3. Gruppierung: Nachfrage pro Store-Item-Kombination pro Woche
demand_per_store_item_week_df = m5_df.groupby(
    ['yearWeek'] + store_columns + item_columns
)['demand_original_reconstructed'].sum().reset_index()

# === 4. Store-Item-Kombination als eindeutige ID
demand_per_store_item_week_df['store_item_id'] = demand_per_store_item_week_df[
    store_columns + item_columns
].apply(lambda x: '_'.join(x.index[x == 1]), axis=1)

# === 5. Entferne One-Hot Spalten zur Übersicht in dem temporären DataFrame
demand_per_store_item_week_df.drop(
    columns=store_columns + item_columns,
    inplace=True
)

# === 6. Durchschnittlicher Demand je Store-Item-Kombination für Sortierung der Item-Store-Kombinationen für Farbskala
store_item_combinations = demand_per_store_item_week_df['store_item_id'].unique()
avg_demand_per_store_item = {
    comb: demand_per_store_item_week_df[demand_per_store_item_week_df['store_item_id'] == comb]['demand_original_reconstructed'].mean()
    for comb in store_item_combinations
}

# === 7. Sortieren nach Durchschnittsdemand
sorted_store_item_combinations_week = sorted(
    avg_demand_per_store_item,
    key=avg_demand_per_store_item.get,
    reverse=True
)

# === 8. Farbskala (z. B. Turbo, invertiert: rot = viel, blau = wenig)
color_scale_name = "Turbo"
color_positions = np.linspace(1, 0, len(sorted_store_item_combinations_week))  # Reversed!
color_scale = px.colors.sample_colorscale(color_scale_name, color_positions)

# === 9. Plot vorbereiten
fig = go.Figure()

for i, store_item in enumerate(sorted_store_item_combinations_week):
    df_store_item = demand_per_store_item_week_df[
        demand_per_store_item_week_df['store_item_id'] == store_item
    ].copy()

    # Konvertiere yearWeek in Datetime (Montag jeder ISO-Woche)
    df_store_item['yearWeek'] = pd.to_datetime(
        df_store_item['yearWeek'] + '-1', format='%G-%V-%u'
    )

    df_store_item.dropna(subset=['demand_original_reconstructed'], inplace=True)

    fig.add_trace(go.Scatter(
        x=df_store_item['yearWeek'].tolist(),
        y=df_store_item['demand_original_reconstructed'].astype(float).tolist(),
        mode='lines',
        name=store_item,
        line=dict(color=color_scale[i])
    ))

# === 10. Layout anpassen
fig.update_layout(
    title='Total demand per store-item combination per week',
    xaxis_title='week',
    yaxis_title='Demand',
    height=600,
    margin=dict(t=50, b=50, l=50, r=250),
    legend=dict(
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.05
    ),
    coloraxis=dict(colorscale=color_scale_name)
)

# === 11. Plot anzeigen
fig.show()

save_plotly_figure_2(fig, 'demand_per_store_item_week', 'perItemStore')

### Item-Store Kombinationen (Heatmaps)

In [ ]:
# Gesamter Absatz pro Store und Item
pivot_df = m5_df.groupby(['store_id', 'item_id'])['demand_original_reconstructed'].sum().unstack(fill_value=0)
# strip name item from the cols starting with 'item_'
pivot_df.columns = pivot_df.columns.str.replace('item_', '', regex=False)
# strip name store from the index starting with 'store_'
pivot_df.index = pivot_df.index.str.replace('store_', '', regex=False)
pivot_df

In [ ]:
plt.figure(figsize=(8, 6))
ax = sns.heatmap(pivot_df, cmap="YlGnBu", annot=True, fmt=".0f")

# Drehe die y-Tick-Labels (Store-Namen) auf 0 Grad (horizontal)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

# Drehe die x-Tick-Labels (Item-Namen) leicht und setze mehr Platz unten
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Mehr Platz unten für die x-Ticks
plt.tight_layout(rect=[0, 0.15, 1, 1])  # [left, bottom, right, top]

plt.title("Gesamter Absatz pro Store und Item")
plt.xlabel("Item")
plt.ylabel("Store")
plt.show()


In [ ]:
# relative demand per store and item
relative_demand = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100  # Prozentuale Verteilung
relative_demand


In [ ]:
# plot the relative demand as a heatmap
plt.figure(figsize=(8, 6))
ax = sns.heatmap(relative_demand, cmap="YlGnBu", annot=True, fmt=".1f")

# Drehe die y-Tick-Labels (Store-Namen) auf 0 Grad (horizontal)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

# Drehe die x-Tick-Labels (Item-Namen) leicht und setze mehr Platz unten
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Mehr Platz unten für die x-Ticks
plt.tight_layout(rect=[0, 0.15, 1, 1])  # [left, bottom, right, top]

plt.title("Relative Nachfrage pro Store und Item")
plt.xlabel("Item")
plt.ylabel("Store")
plt.show()

In [ ]:
m5_df['week'] = m5_df['date'].dt.isocalendar().week # Kalenderwoche Nummerierung, je Jahr beginnt bei 1

# Wochenbasierte Mittelwerte
weekly_avg_df = m5_df.groupby(['store_id', 'item_id', 'year', 'week'])['demand_original_reconstructed'].mean().reset_index()

# Dann wieder pivotieren (z. B. über Mittel aller Wochen)
weekly_avg_pivot_df = weekly_avg_df.groupby(['store_id', 'item_id'])['demand_original_reconstructed'].mean().unstack(fill_value=0)

# strip name item / store from cols / index starting with 'item_' / 'store_'
weekly_avg_pivot_df.columns = weekly_avg_pivot_df.columns.str.replace('item_', '', regex=False)
weekly_avg_pivot_df.index = weekly_avg_pivot_df.index.str.replace('store_', '', regex=False)

In [ ]:
weekly_avg_pivot_df

In [ ]:
# plot the pivot_avg as a heatmap
plt.figure(figsize=(8, 6))
ax = sns.heatmap(weekly_avg_pivot_df, cmap="YlGnBu", annot=True, fmt=".0f")

# Drehe die y-Tick-Labels (Store-Namen) auf 0 Grad (horizontal)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

# Drehe die x-Tick-Labels (Item-Namen) leicht und setze mehr Platz untenS
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Mehr Platz unten für die x-Ticks
plt.tight_layout(rect=[0, 0.15, 1, 1])  # [left, bottom, right, top]

plt.title("Durchschnittlicher Absatz pro Store und Item (Wochenbasis)")
plt.xlabel("Item")
plt.ylabel("Store")
plt.show()

## 5. Feature Emgineering & Transformation

In [ ]:
m5_df.columns

### Zeitbezogenen Variablen

| Feature     | Encoding              | Zweck                  |
| ----------- | --------------------- | ---------------------- |
| `dayIndex`  | linear                | Zeitlicher Trend       |
| `dayofyear` | sin/cos               | Jährliche Saisonalität |
| `weekday`   | sin/cos               | Wöchentliche Zyklen    |
| `month`     | sin/cos               | Monatliche Muster      |
| `year`      | skaliert (z. B. -min) | Langfristiger Trend    |
| `month`     | label                 |       |
| `weekday`   | label                 |     |


In [ ]:
# 1. Month und weekday Label-Encoding
# create weekday und month from the one-hot encoded equivalenten (label encoded)
m5_df['weekday'] = m5_df.filter(like='weekday_').idxmax(axis=1).str.replace('weekday_', '', regex=False)
m5_df['month'] = m5_df.filter(like='month_').idxmax(axis=1).str.replace('month_', '', regex=False)
# ckeck if the columns are created correctly
weekday_month_cols = [col for col in m5_df.columns if col.startswith(('weekday_', 'month_'))]
m5_df[['date'] + weekday_month_cols + ['weekday', 'month']].head()


In [ ]:
# manuell label encoding für weekday und month
# Manuelle Mappings
weekday_order = {'MON': 0, 'TUE': 1, 'WED': 2, 'THU': 3,'FRI': 4, 'SAT': 5, 'SUN': 6}
month_order = {'JAN': 0, 'FEB': 1, 'MAR': 2, 'APR': 3, 'MAY': 4, 'JUN': 5, 'JUL': 6, 'AUG': 7, 'SEP': 8, 'OCT': 9, 'NOV': 10, 'DEC': 11}

# replace the weekday and month columns with the label encoded values
m5_df['weekday'] = m5_df['weekday'].map(weekday_order)
m5_df['month'] = m5_df['month'].map(month_order)
# check if the columns are created correctly
m5_df[['date', 'weekday', 'month'] + weekday_month_cols].head()

In [ ]:
# 2. Year scaled
# print unique values of year
m5_df['year'].unique()

In [ ]:
# year_scaled (lineares Trendfeature)
m5_df['year_scaled'] = m5_df['year'] - m5_df['year'].min()
m5_df['year_scaled'].unique()

In [ ]:
# 3. dayofyear + sin/cos encoding
m5_df['dayofyear'] = m5_df['date'].dt.dayofyear
m5_df['dayofyear_sin'] = np.sin(2 * np.pi * m5_df['dayofyear'] / 365.25)  # 365.25 für Schaltjahre
m5_df['dayofyear_cos'] = np.cos(2 * np.pi * m5_df['dayofyear'] / 365.25)  # 365.25 für Schaltjahre

In [ ]:
# 4. weekday + sin/cos encoding
m5_df['weekday_sin'] = np.sin(2 * np.pi * m5_df['weekday'] / 7)
m5_df['weekday_cos'] = np.cos(2 * np.pi * m5_df['weekday'] / 7)
# 5. month + sin/cos encoding
m5_df['month_sin'] = np.sin(2 * np.pi * m5_df['month'] / 12)
m5_df['month_cos'] = np.cos(2 * np.pi * m5_df['month'] / 12)

In [ ]:
m5_df.head(2)

### Adding Lag Variablen und mehr

In [ ]:
# --- 0) Sortieren, Konstanten ---
m5_df = m5_df.sort_values(['id', 'date']).copy()

LAGS = [1, 7, 14, 28]
MAX_LAG = max(LAGS)

# --- 1) Demand-Lags erstellen (gruppenweise) ---
for k in LAGS:
    col = f'demand_lag_{k}'
    m5_df[col] = m5_df.groupby('id', group_keys=False)['demand'].shift(k)
    # Flag: lag war ursprünglich NaN (vor jeglichem Füllen)
    m5_df[f'{col}_was_nan'] = m5_df[col].isna().astype(int)

# --- 2) Nur aus Vergangenheit füllen (keine Median/Mean-Imputation) ---
lag_cols = [f'demand_lag_{k}' for k in LAGS]
m5_df[lag_cols] = m5_df.groupby('id', group_keys=False)[lag_cols].ffill()

# --- 3) Diffs jetzt aus den (nur-FFill) Lags berechnen ---
for k in LAGS:
    col = f'demand_diff_{k}'
    m5_df[col] = m5_df['demand'] - m5_df[f'demand_lag_{k}']
    # Flag für diff: entspricht dem ursprünglichen lag-NaN-Flag
    m5_df[f'{col}_was_nan'] = m5_df[f'demand_lag_{k}_was_nan']

# --- 4) Rolling-Mittel & Ratios (falls noch nicht vorhanden) ---
for window in [7, 28]:
    rm_col = f'demand_rolling_mean_{window}'
    m5_df[rm_col] = (m5_df
                         .groupby('id', group_keys=False)['demand']
                         .transform(lambda x: x.rolling(window=window, min_periods=1).mean()))
    m5_df[f'demand_ratio_to_{window}_avg'] = m5_df['demand'] / (m5_df[rm_col] + 1e-5)

In [ ]:
# --- 5) Weitere manuelle Lags (nur Vergangenheit), inkl. Wetter (nur bakery) ---
EXTRA_BASES = [
    'demand_rolling_mean_7','demand_rolling_mean_28',
    'demand_ratio_to_7_avg','demand_ratio_to_28_avg',
    'demand__standard_deviation_7','demand__standard_deviation_14','demand__standard_deviation_28',
    'demand__maximum_7','demand__maximum_14','demand__maximum_28'
]
EXTRA_LAGS = [1, 7]

extra_lag_cols = []
for base in EXTRA_BASES:
    if base not in m5_df.columns:  # falls einzelne Stats nicht existieren
        continue
    for k in EXTRA_LAGS:
        c = f'{base}_lag_{k}'
        m5_df[c] = m5_df.groupby('id', group_keys=False)[base].shift(k)
        m5_df[f'{c}_was_nan'] = m5_df[c].isna().astype(int)
        extra_lag_cols.append(c)

# Nur aus Vergangenheit füllen (falls innerhalb der Serie NaNs vorhanden sind)
if extra_lag_cols:
    m5_df[extra_lag_cols] = m5_df.groupby('id', group_keys=False)[extra_lag_cols].ffill()

# --- 6) Burn-in droppen: die ersten MAX_LAG Zeilen je Serie ---
m5_df = (m5_df
             .groupby('id', group_keys=False)
             .apply(lambda g: g.iloc[MAX_LAG:])
             .reset_index(drop=True))

In [ ]:
# --- 7) Sanity-Checks ---
need_no_nan = lag_cols + [f'demand_diff_{k}' for k in LAGS] + extra_lag_cols
nan_summary = m5_df[need_no_nan].isna().sum().sort_values(ascending=False)
print("NaNs in Kern-Lag/Diff/Extra-Lag-Features nach Burn-in:")
print(nan_summary[nan_summary > 0] if (nan_summary > 0).any() else "Keine NaNs mehr in den Kernfeatures.")
print("Shape:", m5_df.shape)

## Speichern des neuen Dataframes

In [ ]:
#m5_df.dtypes

In [ ]:
# save the new dataframe to a new csv file
m5_df.to_csv(Path.cwd().parents[1] / 'data' / 'basic_preprocessed' / 'm5_basic_prepro.csv', index=False)

In [ ]:
import pandas as pd
from pathlib import Path

# ckeck by loading the new dataframe
prepro_df = pd.read_csv(Path.cwd().parents[1] / 'data' / 'basic_preprocessed' / 'm5_basic_prepro.csv')
prepro_df.head(2)

In [ ]:
# compare the shape of the preprocessed dataframe bevor and after saving and reloading
print(f"Original dataframe at end shape: {m5_df.shape}")
print(f"Preprocessed dataframe shape:    {prepro_df.shape}")

In [ ]:
# compare datetype of all columns in the original and preprocessed dataframe
for col in m5_df.columns:
    if m5_df[col].dtype != prepro_df[col].dtype:
        print(f"Column '{col}' has different dtype: {m5_df[col].dtype} vs {prepro_df[col].dtype}")

# Sesonality Check with Darts

In [ ]:
# === Seasonality-Check (gründlich, EDA) ===
from pathlib import Path
import pandas as pd
import numpy as np

from darts import TimeSeries
from darts.utils.statistics import check_seasonality
try:
    from darts.utils.missing_values import fill_missing_values
    DARTS_HAS_MV_UTILS = True
except Exception:
    DARTS_HAS_MV_UTILS = False

In [ ]:
# loading the new dataframe if not already loaded
prepro_df = pd.read_csv(Path.cwd().parents[1] / 'data' / 'basic_preprocessed' / 'm5_basic_prepro.csv')
prepro_df.head(2)

In [ ]:
Path.cwd().parents[1]

In [ ]:
# ----- Konfig -----
DATASET   = "m5"                     # <— bzw. "bakery"
M_LIST    = [7, 14, 28]              # <— zu prüfende Saisonalitäten
VAL_DAYS  = 28
TEST_DAYS = 28

FILL_MISSING = True
FILL_METHOD  = "auto"

PROJECT_ROOT = Path.cwd().parents[1]
OUT_DIR = PROJECT_ROOT / "outputs" / "eda" / DATASET.lower()
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Hilfsfunktionen
# TimeSeries: fehlende Werte füllen (sollten keine fehlen, aber sicher ist sicher), da NaiveSeasonal kein NaN akzeptiert
def ts_has_nan(ts: TimeSeries) -> bool:
    try:    return np.isnan(ts.values(copy=False)).any()
    except: return ts.pd_series().isna().any()

def ts_fill_missing(ts: TimeSeries, method="auto"):
    if not FILL_MISSING or not ts_has_nan(ts): return ts
    if DARTS_HAS_MV_UTILS:
        try: return fill_missing_values(ts, fill=method)
        except: pass
    s = ts.pd_series().ffill().bfill()
    return TimeSeries.from_series(s, freq=ts.freq_str if ts.freq is not None else "D")

# globaler Cut für Trainingsdaten (für alle Serien gleich, damit vergleichbar)
def get_global_train_cut(df, val_days, test_days):
    last_date = df["date"].max()
    return last_date - pd.Timedelta(days=val_days + test_days)

# führende Nullen abschneiden (für MASE/RMSSE), damit diese nicht die Denominators verfälschen (sollte nicht der Fall sein)
def trim_leading_zeros(ts: TimeSeries) -> TimeSeries:
    a = ts.values(copy=False).flatten()
    nz = np.nonzero(a)[0]
    return ts if len(nz)==0 else ts[nz[0]:]

In [ ]:
# Da NaiveSeasonal nur target Spalte braucht, alle anderen entfernen
prepro_df = (prepro_df[["date", "id", "demand"]]
      .drop_duplicates(subset=["id","date"])
      .sort_values(["id","date"])
      .reset_index(drop=True))

prepro_df.head(2)

In [ ]:
prepro_df.dtypes

In [ ]:
# change col date to datetime
prepro_df['date'] = pd.to_datetime(prepro_df['date'])
prepro_df.dtypes

In [ ]:
# prüft ob es Lücken in den Zeitreihen gibt
def quality_checks(df: pd.DataFrame) -> pd.DataFrame:
    stats = []
    for gid, g in df.groupby("id"):
        
        exp = (g['date'].max() - g['date'].min()).days + 1
        actual = g['date'].nunique()
        gaps = exp - actual
        stats.append({
            'id': gid,
            'first': g['date'].min(),
            'last': g['date'].max(),
            'n_points': actual,
            'gaps': gaps,
        })
    qs = pd.DataFrame(stats).sort_values('n_points')
    return qs

qs = quality_checks(prepro_df)
display(qs.head(10))

print("IDs mit Lücken (gaps > 0):", (qs['gaps'] > 0).sum(), "von", len(qs))

In [ ]:
# TimeSeries bauen
series_by_id = {}
for gid, g in prepro_df.groupby("id"):
    ts = TimeSeries.from_dataframe(g, time_col="date", value_cols="demand", freq="D")
    ts = ts_fill_missing(ts, method=FILL_METHOD)
    series_by_id[gid] = ts

print(f"Anzahl Serien: {len(series_by_id)}")

In [ ]:
# Globale Cut-Daten
global_train_cut = get_global_train_cut(prepro_df, VAL_DAYS, TEST_DAYS)
print(f"Val days: {VAL_DAYS}, Test days: {TEST_DAYS}\n")
print(f"Trainingsdaten beginnen am:  {prepro_df['date'].min()}")
print(f"Globaler Trainingsdaten-Cut: {global_train_cut}")
print(f"Trainingsdaten enden am:     {global_train_cut}")
print(f"Validierung bis:             {global_train_cut + pd.Timedelta(days=VAL_DAYS)}")
print(f"Test bis:                    {global_train_cut + pd.Timedelta(days=VAL_DAYS + TEST_DAYS)}")

In [ ]:
# Hilfsfunktion: DataFrame → TimeSeries (mit Missing-Value-Füllung)
def _to_ts(g):
    ts = TimeSeries.from_dataframe(g, time_col="date", value_cols="demand", freq="D")
    if DARTS_HAS_MV_UTILS:
        try:
            ts = fill_missing_values(ts, fill="auto")
        except Exception:
            s = ts.pd_series().ffill().bfill()
            ts = TimeSeries.from_series(s, freq="D")
    else:
        s = ts.pd_series().ffill().bfill()
        ts = TimeSeries.from_series(s, freq="D")
    return ts

In [ ]:
# ----- Seasonality-Check -----
rows = []
for gid, g in prepro_df.groupby("id"):
    ts = _to_ts(g)
    train, _ = ts.split_after(global_train_cut)
    n = len(train)

    rec = {"id": gid, "n_train": n}
    for m in M_LIST:
        L = 4*m                       # fixe Ziel-Obergrenze (global konsistent)
        eff_max_lag = min(n-1, L)
        if eff_max_lag < m:
            is_seas, eff_m = (False, None)   # oder np.nan, wenn du „nicht beurteilbar“ markieren willst
        else:
            try:
                is_seas, eff_m = check_seasonality(train, m=m, max_lag=eff_max_lag)
            except Exception:
                is_seas, eff_m = (False, None)

        rec[f"is_seasonal_m{m}"] = bool(is_seas)
        rec[f"eff_m{m}"] = int(eff_m) if eff_m is not None else None
        rec[f"eff_max_lag_m{m}"] = eff_max_lag
    rows.append(rec)

flags = pd.DataFrame(rows)

# Speichern
fname = f"seasonality_flags_{DATASET.lower()}_traincut_val{VAL_DAYS}_test{TEST_DAYS}.csv"
out_csv = OUT_DIR / fname
flags.to_csv(out_csv, index=False)
print("Seasonality-Flags gespeichert nach:", out_csv)

# Kurze Zusammenfassung
for m in M_LIST:
    total = len(flags)
    n_true = flags[f"is_seasonal_m{m}"].sum()
    print(f"m={m}: {n_true}/{total} Serien (~{100*n_true/total:.1f}%) mit signifikanter Saisonalität")

display(flags.head())

### Visualisierungs Ideen

In [ ]:
# === Setup für Plots ===
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DATASET = "bakery"  # ggf. anpassen
M_LIST  = [7,14,28] # muss zu deinen Spalten in `flags` passen

PLOT_DIR = Path.cwd().parents[1] / "outputs" / "eda" / DATASET / "figs"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Long-Format bauen
records = []
for m in M_LIST:
    dfm = flags[['id','n_train',f'is_seasonal_m{m}',f'eff_m{m}',f'eff_max_lag_m{m}']].copy()
    dfm['m'] = m
    dfm = dfm.rename(columns={f'is_seasonal_m{m}':'is_seasonal',
                              f'eff_m{m}':'eff_m',
                              f'eff_max_lag_m{m}':'eff_max_lag'})
    records.append(dfm)
long = pd.concat(records, ignore_index=True)


In [ ]:
share = (long.groupby('m')['is_seasonal']
              .value_counts(normalize=True)
              .rename('share')
              .reset_index())

fig, ax = plt.subplots(figsize=(6,4))
for i, m in enumerate(sorted(M_LIST)):
    s = share[share['m']==m]
    yes = float(s[s['is_seasonal']==True]['share']) if any((s['is_seasonal']==True)) else 0.0
    ax.bar(i, 1.0, label=None)             # 100%
    ax.bar(i, yes, label=None)             # Anteil True übermalen
    ax.text(i, 0.5, f"{yes*100:.0f}%", ha='center', va='center', color='white', weight='bold')
ax.set_xticks(range(len(M_LIST))); ax.set_xticklabels([f"m={m}" for m in M_LIST])
ax.set_ylim(0,1); ax.set_ylabel("Anteil (True)")
ax.set_title("Saisonalität erkannt (Anteil je m)")
plt.tight_layout()
plt.savefig(PLOT_DIR / "seasonality_share_per_m.png", dpi=150)
plt.show()


In [ ]:
mat = (long.pivot(index='id', columns='m', values='is_seasonal')
           .reindex(columns=sorted(M_LIST)))
# IDs optional nach n_train sortieren:
id_order = (long.groupby('id')['n_train'].max()
                 .sort_values(ascending=False).index)
mat = mat.loc[id_order]
mat_int = mat.astype(float)  # True=1.0, False=0.0, NaN bleibt NaN

fig, ax = plt.subplots(figsize=(6, max(4, len(mat_int)/10)))
im = ax.imshow(mat_int.values, aspect='auto')
ax.set_xticks(range(len(M_LIST))); ax.set_xticklabels([f"m={m}" for m in M_LIST])
ax.set_yticks([])  # viele IDs -> Achse aus
ax.set_title("Saisonalität je ID und m (True=1, False=0)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(PLOT_DIR / "seasonality_heatmap_id_by_m.png", dpi=150)
plt.show()


In [ ]:
# count id x m combinations for all True, all False, mixed
def count_id_m_combinations(df):
    counts = {'all_true': 0, 'all_false': 0, 'mixed': 0}
    for gid, g in df.groupby('id'):
        vals = g['is_seasonal'].dropna().unique()
        if len(vals) == 1:
            if vals[0] == True:
                counts['all_true'] += 1
            else:
                counts['all_false'] += 1
        elif len(vals) > 1:
            counts['mixed'] += 1
    return counts

counts = count_id_m_combinations(long)
print("Anzahl IDs mit:")
for k, v in counts.items():
    print(f"  {k}: {v}")

In [ ]:
# list all id x m combis that are seasonal
seasonal_combis = long[long['is_seasonal'] == True][['id', 'm', 'eff_m', 'n_train']]
print(f"Anzahl saisonaler ID x m-Kombis: {len(seasonal_combis)}")
display(seasonal_combis.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for m in sorted(M_LIST):
    dfm = long[long['m']==m]
    # Jitter auf y, damit Punkte pro m nicht exakt übereinander liegen
    y = np.full(len(dfm), m) + (np.random.rand(len(dfm))-0.5)*0.3
    ax.scatter(dfm['n_train'], y, s=12, alpha=0.6,
               label=f"m={m}")
ax.set_xlabel("n_train")
ax.set_ylabel("m (leicht gejittert)")
ax.set_title("Train-Länge je ID über m")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "seasonality_scatter_ntrain_by_m.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(M_LIST), figsize=(4*len(M_LIST),4), sharey=True)
if len(M_LIST)==1:
    axes = [axes]
for ax, m in zip(axes, sorted(M_LIST)):
    dfm = long[long['m']==m].copy()
    dfm['flag'] = np.where(dfm['is_seasonal'], 'seasonal', 'non-seasonal')
    groups = [dfm[dfm['flag']=='non-seasonal']['n_train'],
              dfm[dfm['flag']=='seasonal']['n_train']]
    ax.boxplot(groups, labels=['non','seasonal'])
    ax.set_title(f"m={m}")
    ax.set_xlabel("Flag"); ax.set_ylabel("n_train")
fig.suptitle("Train-Länge vs. Saisonalität")
plt.tight_layout()
plt.savefig(PLOT_DIR / "seasonality_boxplot_ntrain_by_flag.png", dpi=150)
plt.show()
